In [1]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline


/home/lifeweb/.local/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [3]:
x_train = train_df.drop(columns=["log_target_price"]).copy()
y_train = train_df["log_target_price"]
x_test = train_df.drop(columns=["log_target_price"]).copy()
y_test = train_df["log_target_price"]

In [4]:
x_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 684678 entries, 0 to 684677
Data columns (total 14 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   category           684678 non-null  int64  
 1   city_slug          684678 non-null  int64  
 2   building_size      684678 non-null  float64
 3   deed_type          684678 non-null  int64  
 4   has_business_deed  684678 non-null  int64  
 5   rooms_count        684678 non-null  int64  
 6   has_elevator       684678 non-null  int64  
 7   construction_year  684678 non-null  float64
 8   is_rebuilt         684678 non-null  int64  
 9   luxury_features    684678 non-null  float64
 10  basic_features     684678 non-null  float64
 11  floor_ratio        684678 non-null  float64
 12  utm_x              684678 non-null  float64
 13  utm_y              684678 non-null  float64
dtypes: float64(7), int64(7)
memory usage: 73.1 MB


In [5]:
rf_params = {
    "n_estimators": [300],
    "min_samples_split": [50, 100],
    "max_depth": [None, 25]
}

rf_grid = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), rf_params, cv=3, scoring="neg_mean_squared_error", n_jobs=-1)
rf_grid.fit(x_train, y_train)
best_rf_model = rf_grid.best_estimator_
print(rf_grid.best_params_)

{'max_depth': None, 'min_samples_split': 100, 'n_estimators': 300}


In [6]:
save_path_rf = "random_forest.joblib"
joblib.dump(best_rf_model, save_path_rf)

['random_forest.joblib']

In [7]:
xgb_params = {
    "n_estimators": [None, 200, 400],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.8, 1]}

xgb_grid = GridSearchCV(XGBRegressor(random_state=42, objective="reg:squarederror"), xgb_params, cv=3, scoring="neg_mean_squared_error", n_jobs=-1)
xgb_grid.fit(x_train, y_train)
best_xgb_model = xgb_grid.best_estimator_
print(xgb_grid.best_params_)

{'learning_rate': 0.1, 'n_estimators': 400, 'subsample': 1}


In [8]:
save_path_xgb = "xgboost.joblib"
joblib.dump(best_xgb_model, save_path_xgb)

['xgboost.joblib']

In [9]:
poly_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("lr", LinearRegression())])

poly_param = {"poly__degree": [1, 2, 3]}
poly_grid = GridSearchCV(poly_model, poly_param, cv=3, scoring="neg_mean_squared_error",n_jobs=-1)
poly_grid.fit(x_train, y_train)
best_poly_model = poly_grid.best_estimator_
print(poly_grid.best_params_)

{'poly__degree': 3}


In [10]:
save_path_poly = "polynomial_regression.joblib"
joblib.dump(best_poly_model, save_path_poly)

['polynomial_regression.joblib']

In [11]:
models = {
    "RandomForestRegressor": best_rf_model,
    "XGBRegressor": best_xgb_model,
    "PolyRegression":best_poly_model
}

In [12]:
results = []
for name, model in models.items():
    y_pred = model.predict(x_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    
    results.append({
        "Model": name,
        "R2": r2,
        "MAE": mae,
        "MSE": mse
    })

In [14]:
report = pd.DataFrame(results)
report

,Model,R2,MAE,MSE
0,RandomForestRegressor,0.319117,0.608254,1.914060
1,XGBRegressor,0.223771,0.659602,2.182091
2,PolyRegression,0.123187,0.752471,2.464848
